In [ ]:
!pip install streamlit pyjwt bcrypt python-dotenv pyngrok nltk streamlit-option-menu plotly textstat PyPDF2 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.3/829.3 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 47.3 MB/s eta 0:00:00


In [ ]:
%%writefile db.py
import sqlite3
import hashlib
import bcrypt
import datetime
import time
import random
import string
from contextlib import contextmanager

# ================= CONFIGURATION =================
DB_NAME = "users.db"
MAX_ATTEMPTS = 3
LOCKOUT_SECONDS = 60
OTP_EXPIRY_SECONDS = 300  # 5 minutes

# ================= DATABASE CONNECTION MANAGER =================
@contextmanager
def get_db_connection():
    conn = None
    try:
        conn = sqlite3.connect(DB_NAME, timeout=10)
        conn.execute("PRAGMA journal_mode=WAL")
        yield conn
        conn.commit()
    except Exception as e:
        if conn:
            conn.rollback()
        raise e
    finally:
        if conn:
            conn.close()

def execute_query(query, params=(), fetch_one=False, fetch_all=False):
    try:
        with get_db_connection() as conn:
            cur = conn.cursor()
            cur.execute(query, params)
            if fetch_one:
                return cur.fetchone()
            elif fetch_all:
                return cur.fetchall()
            else:
                conn.commit()
                return True
    except Exception as e:
        print(f"Database error: {str(e)}")
        return None if (fetch_one or fetch_all) else False

# ================= TIMESTAMP HELPER =================
def _get_timestamp():
    return datetime.datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")

def get_relative_time(date_str):
    if not date_str:
        return "some time ago"
    try:
        past = datetime.datetime.strptime(date_str, "%Y-%m-%d %H:%M:%S")
        diff = datetime.datetime.utcnow() - past
        days = diff.days
        seconds = diff.seconds
        if days > 365:
            return f"{days // 365} years ago"
        elif days > 30:
            return f"{days // 30} months ago"
        elif days > 0:
            return f"{days} days ago"
        elif seconds > 3600:
            return f"{seconds // 3600} hours ago"
        elif seconds > 60:
            return f"{seconds // 60} minutes ago"
        else:
            return "just now"
    except:
        return date_str

# ================= OTP FUNCTIONS =================
def generate_otp():
    """Generate a 6-digit OTP"""
    return ''.join(random.choices(string.digits, k=6))

def save_otp(email, otp):
    """Save OTP to database"""
    try:
        expiry = time.time() + OTP_EXPIRY_SECONDS
        with get_db_connection() as conn:
            c = conn.cursor()
            # Create OTP table if not exists
            c.execute('''CREATE TABLE IF NOT EXISTS otp_verification (
                email TEXT PRIMARY KEY,
                otp TEXT NOT NULL,
                expiry REAL NOT NULL,
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                FOREIGN KEY(email) REFERENCES users(email) ON DELETE CASCADE
            )''')

            # Insert or replace OTP
            c.execute('''
                INSERT OR REPLACE INTO otp_verification (email, otp, expiry)
                VALUES (?, ?, ?)
            ''', (email, otp, expiry))
            conn.commit()
            return True
    except Exception as e:
        print(f"Error saving OTP: {str(e)}")
        return False

def verify_otp(email, otp):
    """Verify OTP"""
    try:
        with get_db_connection() as conn:
            c = conn.cursor()
            c.execute('''
                SELECT otp, expiry FROM otp_verification
                WHERE email = ? AND otp = ?
            ''', (email, otp))
            result = c.fetchone()

            if result:
                stored_otp, expiry = result
                if time.time() <= expiry:
                    # Delete used OTP
                    c.execute("DELETE FROM otp_verification WHERE email = ?", (email,))
                    conn.commit()
                    return True
            return False
    except Exception as e:
        print(f"Error verifying OTP: {str(e)}")
        return False

def clear_otp(email):
    """Clear OTP for user"""
    try:
        execute_query("DELETE FROM otp_verification WHERE email = ?", (email,))
        return True
    except Exception as e:
        print(f"Error clearing OTP: {str(e)}")
        return False

# ================= PASSWORD HASHING =================
def hash_password(password):
    """Hash password using SHA-256 (for compatibility with original code)"""
    return hashlib.sha256(password.encode()).hexdigest()

def bcrypt_hash_password(password):
    """Hash password using bcrypt (for enhanced security)"""
    salt = bcrypt.gensalt()
    return bcrypt.hashpw(password.encode('utf-8'), salt)

def bcrypt_check_password(password, hashed):
    """Check password against bcrypt hash"""
    return bcrypt.checkpw(password.encode('utf-8'), hashed)

# ================= DATABASE INITIALIZATION =================
def init_db():
    """Initialize the database with all required tables"""
    try:
        with get_db_connection() as conn:
            c = conn.cursor()

            # Users table (with SHA-256 for compatibility)
            c.execute('''CREATE TABLE IF NOT EXISTS users (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                username TEXT NOT NULL,
                email TEXT NOT NULL UNIQUE,
                password TEXT NOT NULL,
                security_question TEXT NOT NULL,
                security_answer TEXT NOT NULL,
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                last_login TIMESTAMP,
                login_count INTEGER DEFAULT 0
            )''')

            # Password History table (with bcrypt for enhanced security)
            c.execute('''CREATE TABLE IF NOT EXISTS password_history (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                email TEXT NOT NULL,
                password BLOB NOT NULL,
                set_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                FOREIGN KEY(email) REFERENCES users(email) ON DELETE CASCADE
            )''')

            # Login Attempts table (Rate Limiting)
            c.execute('''CREATE TABLE IF NOT EXISTS login_attempts (
                email TEXT PRIMARY KEY,
                attempts INTEGER DEFAULT 0,
                last_attempt REAL,
                FOREIGN KEY(email) REFERENCES users(email) ON DELETE CASCADE
            )''')

            # OTP Verification table
            c.execute('''CREATE TABLE IF NOT EXISTS otp_verification (
                email TEXT PRIMARY KEY,
                otp TEXT NOT NULL,
                expiry REAL NOT NULL,
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                FOREIGN KEY(email) REFERENCES users(email) ON DELETE CASCADE
            )''')

            conn.commit()
            return True
    except Exception as e:
        print(f"Failed to initialize database: {str(e)}")
        return False

# ================= USER MANAGEMENT =================
def save_user(username, email, password, security_question, security_answer):
    """Save a new user to the database (for original PolicyNav Pro)"""
    try:
        with get_db_connection() as conn:
            c = conn.cursor()

            # Check if user exists
            c.execute("SELECT email FROM users WHERE email = ?", (email,))
            if c.fetchone():
                return False, "Email already exists"

            # Hash password and answer using SHA-256 (for compatibility)
            hashed_password = hash_password(password)
            hashed_answer = hash_password(security_answer.lower().strip())
            now = _get_timestamp()

            # Insert user
            c.execute("""
                INSERT INTO users
                (username, email, password, security_question, security_answer, created_at, login_count)
                VALUES (?, ?, ?, ?, ?, ?, 0)
            """, (username, email, hashed_password, security_question, hashed_answer, now))

            # Also store in password history with bcrypt for future security
            bcrypt_hashed = bcrypt_hash_password(password)
            c.execute("""
                INSERT INTO password_history (email, password, set_at)
                VALUES (?, ?, ?)
            """, (email, bcrypt_hashed, now))

            conn.commit()
            return True, "User created successfully!"
    except sqlite3.IntegrityError:
        return False, "Email already exists"
    except Exception as e:
        return False, f"Database error: {str(e)}"

def register_user(email, password):
    """Register a new user (for Infosys LLM enhanced version)"""
    try:
        with get_db_connection() as conn:
            c = conn.cursor()

            # Check if user exists
            c.execute("SELECT email FROM users WHERE email = ?", (email,))
            if c.fetchone():
                return False

            # Generate username from email
            username = email.split('@')[0]

            # Hash password with SHA-256 (for compatibility)
            hashed_password = hash_password(password)

            # Default security question/answer (can be updated later)
            default_question = "What is your email?"
            default_answer = hash_password(email.lower().strip())

            now = _get_timestamp()

            # Insert user
            c.execute("""
                INSERT INTO users
                (username, email, password, security_question, security_answer, created_at, login_count)
                VALUES (?, ?, ?, ?, ?, ?, 0)
            """, (username, email, hashed_password, default_question, default_answer, now))

            # Store in password history with bcrypt
            bcrypt_hashed = bcrypt_hash_password(password)
            c.execute("""
                INSERT INTO password_history (email, password, set_at)
                VALUES (?, ?, ?)
            """, (email, bcrypt_hashed, now))

            conn.commit()
            return True
    except sqlite3.IntegrityError:
        return False
    except Exception as e:
        print(f"Registration error: {str(e)}")
        return False

def get_user(email):
    """Get user by email (returns username, password, login_count)"""
    result = execute_query(
        "SELECT username, password, COALESCE(login_count, 0) FROM users WHERE email=?",
        (email,),
        fetch_one=True
    )
    return result

def check_user_exists(email):
    """Check if user exists in database"""
    result = execute_query(
        "SELECT 1 FROM users WHERE email = ?",
        (email,),
        fetch_one=True
    )
    return result is not None

def authenticate_user(email, password):
    """Authenticate user with email and password"""
    try:
        with get_db_connection() as conn:
            c = conn.cursor()
            c.execute("SELECT password FROM users WHERE email = ?", (email,))
            data = c.fetchone()

            if data:
                stored_hash = data[0]
                # Check with SHA-256
                if stored_hash == hash_password(password):
                    _reset_attempts(email)
                    return True

            # If SHA-256 fails, check bcrypt history (for migrated users)
            c.execute("SELECT password FROM password_history WHERE email = ? ORDER BY set_at DESC LIMIT 1", (email,))
            history_data = c.fetchone()
            if history_data and bcrypt_check_password(password, history_data[0]):
                # Update the user's password hash to SHA-256 for future logins
                new_hash = hash_password(password)
                c.execute("UPDATE users SET password = ? WHERE email = ?", (new_hash, email))
                conn.commit()
                _reset_attempts(email)
                return True

            _record_failed_attempt(email)
            return False
    except Exception as e:
        print(f"Authentication error: {str(e)}")
        _record_failed_attempt(email)
        return False

# ================= SECURITY QUESTION FUNCTIONS =================
def get_question(email):
    """Get security question for user"""
    result = execute_query(
        "SELECT security_question FROM users WHERE email=?",
        (email,),
        fetch_one=True
    )
    return result[0] if result else None

def check_answer(email, answer):
    """Check if security answer is correct"""
    result = execute_query(
        "SELECT id FROM users WHERE email=? AND security_answer=?",
        (email, hash_password(answer.lower().strip())),
        fetch_one=True
    )
    return result is not None

# ================= PASSWORD MANAGEMENT =================
def update_password(email, new_password):
    """Update user password"""
    try:
        with get_db_connection() as conn:
            c = conn.cursor()

            # Update SHA-256 hash
            hashed_password = hash_password(new_password)
            c.execute("UPDATE users SET password = ? WHERE email = ?", (hashed_password, email))

            # Add to password history with bcrypt
            now = _get_timestamp()
            bcrypt_hashed = bcrypt_hash_password(new_password)
            c.execute("""
                INSERT INTO password_history (email, password, set_at)
                VALUES (?, ?, ?)
            """, (email, bcrypt_hashed, now))

            conn.commit()
            return True
    except Exception as e:
        print(f"Password update error: {str(e)}")
        return False

def check_password_reused(email, new_password):
    """Check if password has been used before"""
    try:
        with get_db_connection() as conn:
            c = conn.cursor()
            c.execute("SELECT password FROM password_history WHERE email = ? ORDER BY set_at DESC LIMIT 5", (email,))
            history = c.fetchall()

            for (stored_hash,) in history:
                if bcrypt_check_password(new_password, stored_hash):
                    return True
            return False
    except Exception as e:
        print(f"Password reuse check error: {str(e)}")
        return False

def check_is_old_password(email, password):
    """Check if password is an old password and return when it was used"""
    try:
        with get_db_connection() as conn:
            c = conn.cursor()
            c.execute("SELECT password, set_at FROM password_history WHERE email = ? ORDER BY set_at DESC", (email,))
            history = c.fetchall()

            for stored_hash, set_at in history:
                if bcrypt_check_password(password, stored_hash):
                    return set_at
            return None
    except Exception as e:
        print(f"Old password check error: {str(e)}")
        return None

# ================= LOGIN STATISTICS =================
def update_login_stats(email):
    """Update login statistics"""
    try:
        now = _get_timestamp()
        execute_query(
            """UPDATE users SET
               last_login = ?,
               login_count = COALESCE(login_count, 0) + 1
               WHERE email=?""",
            (now, email)
        )
        return True
    except Exception as e:
        print(f"Login stats update error: {str(e)}")
        return False

# ================= RATE LIMITING =================
def _record_failed_attempt(email):
    """Record a failed login attempt"""
    try:
        with get_db_connection() as conn:
            c = conn.cursor()
            now = time.time()

            c.execute("SELECT attempts, last_attempt FROM login_attempts WHERE email = ?", (email,))
            row = c.fetchone()

            if row:
                attempts, last = row
                if now - last > LOCKOUT_SECONDS:
                    c.execute("UPDATE login_attempts SET attempts = 1, last_attempt = ? WHERE email = ?", (now, email))
                else:
                    c.execute("UPDATE login_attempts SET attempts = ?, last_attempt = ? WHERE email = ?", (attempts + 1, now, email))
            else:
                c.execute("INSERT INTO login_attempts (email, attempts, last_attempt) VALUES (?, 1, ?)", (email, now))

            conn.commit()
    except Exception as e:
        print(f"Failed attempt record error: {str(e)}")

def _reset_attempts(email):
    """Reset failed attempts for a user"""
    try:
        execute_query("DELETE FROM login_attempts WHERE email = ?", (email,))
    except Exception as e:
        print(f"Reset attempts error: {str(e)}")

def is_rate_limited(email):
    """Check if user is rate limited"""
    try:
        with get_db_connection() as conn:
            c = conn.cursor()
            c.execute("SELECT attempts, last_attempt FROM login_attempts WHERE email = ?", (email,))
            row = c.fetchone()

            if row:
                attempts, last = row
                elapsed = time.time() - last
                if attempts >= MAX_ATTEMPTS and elapsed < LOCKOUT_SECONDS:
                    return True, LOCKOUT_SECONDS - elapsed
            return False, 0
    except Exception as e:
        print(f"Rate limit check error: {str(e)}")
        return False, 0

# ================= ADMIN FUNCTIONS =================
def get_all_users():
    """Get all users for admin panel"""
    result = execute_query(
        "SELECT email, created_at FROM users ORDER BY created_at DESC",
        fetch_all=True
    )
    return result if result else []

def delete_user(email):
    """Delete a user and all related data"""
    try:
        with get_db_connection() as conn:
            c = conn.cursor()
            c.execute("DELETE FROM password_history WHERE email = ?", (email,))
            c.execute("DELETE FROM login_attempts WHERE email = ?", (email,))
            c.execute("DELETE FROM otp_verification WHERE email = ?", (email,))
            c.execute("DELETE FROM users WHERE email = ?", (email,))
            conn.commit()
            return True
    except Exception as e:
        print(f"Delete user error: {str(e)}")
        return False

# ================= USER STATISTICS =================
def get_user_stats():
    """Get user statistics"""
    try:
        with get_db_connection() as conn:
            c = conn.cursor()

            # Total users
            c.execute("SELECT COUNT(*) FROM users")
            total_users = c.fetchone()[0]

            # Users logged in today
            today = datetime.datetime.utcnow().strftime("%Y-%m-%d")
            c.execute("SELECT COUNT(*) FROM users WHERE last_login LIKE ?", (f"{today}%",))
            active_today = c.fetchone()[0]

            return {
                "total_users": total_users,
                "active_today": active_today
            }
    except Exception as e:
        print(f"User stats error: {str(e)}")
        return {"total_users": 0, "active_today": 0}

# Initialize database when module is imported
init_db()

Writing db.py


In [ ]:
%%writefile readability.py
import textstat

class ReadabilityAnalyzer:
    def __init__(self, text):
        self.text = text
        self.num_sentences = textstat.sentence_count(text)
        self.num_words = textstat.lexicon_count(text, removepunct=True)
        self.num_syllables = textstat.syllable_count(text)
        self.complex_words = textstat.difficult_words(text)
        self.char_count = textstat.char_count(text)

    def get_all_metrics(self):
        return {
            "Flesch Reading Ease": textstat.flesch_reading_ease(self.text),
            "Flesch-Kincaid Grade": textstat.flesch_kincaid_grade(self.text),
            "SMOG Index": textstat.smog_index(self.text),
            "Gunning Fog": textstat.gunning_fog(self.text),
            "Coleman-Liau": textstat.coleman_liau_index(self.text)
        }


Writing readability.py


In [ ]:
%%writefile app.py
import streamlit as st
import jwt
import datetime
import re
import time
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from db import (
    save_user, get_user, update_login_stats, get_question, check_answer,
    update_password, check_user_exists, generate_otp, save_otp, verify_otp,
    hash_password
)
import PyPDF2
import readability
import plotly.graph_objects as go

# ================= CONFIG =================
SECRET_KEY = "super_secret_key_for_demo_2024_enhanced"
ALGORITHM = "HS256"
ACCESS_TOKEN_EXPIRE_MINUTES = 30

# Email Configuration
EMAIL_ADDRESS = "saviyadav2006@gmail.com"  # Change this
EMAIL_APP_PASSWORD = "dsii mpzw nijn xaid"  # Change this to EMAIL_APP_PASSWORD

# ================= EMAIL FUNCTIONS =================
def send_otp_email(recipient_email, otp):
    """Send OTP via email"""
    try:
        # Create message
        msg = MIMEMultipart()
        msg['From'] = EMAIL_ADDRESS
        msg['To'] = recipient_email
        msg['Subject'] = "Password Reset OTP - PolicyNav Pro"

        # Email body
        body = f"""
        <html>
        <body style="font-family: Arial, sans-serif; padding: 20px;">
            <h2 style="color: #667eea;">PolicyNav Pro - Password Reset</h2>
            <p>You requested to reset your password. Use the following OTP to proceed:</p>
            <div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                        padding: 15px; border-radius: 10px; text-align: center; margin: 20px 0;">
                <h1 style="color: white; font-size: 36px; letter-spacing: 5px;">{otp}</h1>
            </div>
            <p>This OTP is valid for 5 minutes.</p>
            <p>If you didn't request this, please ignore this email.</p>
            <br>
            <p>Best regards,<br>PolicyNav Pro Team</p>
        </body>
        </html>
        """

        msg.attach(MIMEText(body, 'html'))

        # Send email
        server = smtplib.SMTP('smtp.gmail.com', 587)
        server.starttls()
        server.login(EMAIL_ADDRESS, EMAIL_APP_PASSWORD)
        server.send_message(msg)
        server.quit()

        return True
    except Exception as e:
        print(f"Email error: {str(e)}")
        return False

# ================= JWT =================
def create_token(email, username):
    payload = {
        "sub": email,
        "username": username,
        "exp": datetime.datetime.utcnow() + datetime.timedelta(minutes=ACCESS_TOKEN_EXPIRE_MINUTES)
    }
    return jwt.encode(payload, SECRET_KEY, algorithm=ALGORITHM)

def verify_token(token):
    try:
        return jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])
    except:
        return None

# ================= SESSION =================
if "jwt" not in st.session_state:
    st.session_state.jwt = None
if "page" not in st.session_state:
    st.session_state.page = "login"
if "reset" not in st.session_state:
    st.session_state.reset = None
if "q" not in st.session_state:
    st.session_state.q = None
if "reset_method" not in st.session_state:
    st.session_state.reset_method = None
if "otp_sent" not in st.session_state:
    st.session_state.otp_sent = False
if "otp_verified" not in st.session_state:
    st.session_state.otp_verified = False
if "chat_input_key" not in st.session_state:
    st.session_state.chat_input_key = 0
if "selected_tab" not in st.session_state:
    st.session_state.selected_tab = "Chat"  # Default tab

# ================= UI CONFIG =================
st.set_page_config(
    page_title="PolicyNav Pro",
    page_icon="🔐",
    layout="wide",
    initial_sidebar_state="expanded"
)

# ================= CUSTOM CSS =================
st.markdown("""
<style>
    /* Import Google Fonts */
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&display=swap');
    @import url('https://fonts.googleapis.com/css2?family=Poppins:wght@300;400;500;600;700&display=swap');

    /* Global Styles */
    * {
        font-family: 'Inter', sans-serif;
    }

    /* Main app background */
    .stApp {
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
    }

    /* ===== SIDEBAR STYLING ===== */
    section[data-testid="stSidebar"] {
        background: linear-gradient(135deg, #1a1f36 0%, #2d3748 100%) !important;
        border-right: 1px solid rgba(255,255,255,0.1);
    }

    section[data-testid="stSidebar"] .stMarkdown,
    section[data-testid="stSidebar"] h1,
    section[data-testid="stSidebar"] h2,
    section[data-testid="stSidebar"] h3,
    section[data-testid="stSidebar"] h4,
    section[data-testid="stSidebar"] p {
        color: white !important;
    }

    /* Avatar container */
    .sidebar-avatar {
        width: 80px;
        height: 80px;
        border-radius: 50%;
        background: linear-gradient(135deg, #667eea, #764ba2);
        display: flex;
        align-items: center;
        justify-content: center;
        font-size: 2.5rem;
        font-weight: 600;
        color: white;
        margin: 0 auto 1rem;
        border: 3px solid rgba(255,255,255,0.3);
        box-shadow: 0 4px 15px rgba(0,0,0,0.3);
    }

    /* Sidebar menu options */
    .sidebar-option {
        padding: 12px 15px;
        margin: 5px 0;
        border-radius: 10px;
        cursor: pointer;
        color: rgba(255,255,255,0.8);
        transition: all 0.3s ease;
        font-weight: 500;
    }

    .sidebar-option:hover {
        background: rgba(102, 126, 234, 0.3);
        color: white;
    }

    .sidebar-option.active {
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        color: white;
        box-shadow: 0 4px 15px rgba(102, 126, 234, 0.4);
    }

    /* ===== MAIN CONTENT STYLING ===== */
    /* Project Title - Enhanced with vibrant gradient */
    .project-title {
        text-align: center;
        color: black;
        font-size: 4rem;
        font-weight: 800;
        margin: 0.5rem 0 0.2rem 0;
        text-shadow: 0 4px 20px rgba(0,0,0,0.3);
        letter-spacing: 3px;
        font-family: 'Poppins', sans-serif;

        /* Vibrant Blue to Purple Gradient */
        background: linear-gradient(135deg,
            #00C9FF 0%,
            #4facfe 50%,
            #00f2fe 100%);
        -webkit-background-clip: text;
        -webkit-text-fill-color: transparent;
        background-clip: text;
        background-size: 200% 200%;
        animation: titleGradient 5s ease infinite;
    }

    @keyframes titleGradient {
        0% { background-position: 0% 50%; }
        50% { background-position: 100% 50%; }
        100% { background-position: 0% 50%; }
    }

    .project-subtitle {
        text-align: center;
        color: rgba(255,255,255,0.95);
        font-size: 1.4rem;
        margin-bottom: 1.5rem;
        text-shadow: 0 2px 8px rgba(0,0,0,0.2);
        font-weight: 400;
        letter-spacing: 1px;
    }

    .project-tagline {
        text-align: center;
        color: rgba(255,255,255,0.8);
        font-size: 1.1rem;
        margin-bottom: 2rem;
        font-style: italic;
    }

    /* Auth Container */
    .auth-container {
        max-width: 450px;
        margin: 0 auto;
        padding: 1rem;
        background: rgba(255, 255, 255, 0.1);
        backdrop-filter: blur(10px);
        border-radius: 20px;
        border: 1px solid rgba(255,255,255,0.2);
        box-shadow: 0 15px 35px rgba(0,0,0,0.2);
    }

    /* Auth Headers */
    .auth-header {
        color: white !important;
        text-align: center;
        margin-bottom: 0.5rem;
        font-size: 2.2rem !important;
        font-weight: 700 !important;
        text-shadow: 0 2px 5px rgba(0,0,0,0.2);
    }

    .auth-subtitle {
        text-align: center;
        color: rgba(255,255,255,0.9);
        margin-bottom: 1.5rem;
        font-size: 1rem;
    }

    /* Input Fields */
    .stTextInput > div > div > input {
        background: rgba(255, 255, 255, 0.95) !important;
        border: 1px solid rgba(255,255,255,0.3) !important;
        border-radius: 12px !important;
        padding: 12px 16px !important;
        color: #1f2937 !important;
        box-shadow: 0 4px 10px rgba(0,0,0,0.1);
    }

    /* Text Area for Readability */
    .stTextArea > div > div > textarea {
        background: rgba(255, 255, 255, 0.95) !important;
        border: 1px solid rgba(255,255,255,0.3) !important;
        border-radius: 12px !important;
        padding: 12px 16px !important;
        color: #1f2937 !important;
        box-shadow: 0 4px 10px rgba(0,0,0,0.1);
    }

    /* Selectbox */
    .stSelectbox > div > div > div {
        background: rgba(255, 255, 255, 0.95) !important;
        border: 1px solid rgba(255,255,255,0.3) !important;
        border-radius: 12px !important;
        padding: 8px 12px !important;
    }

    /* Buttons */
    .stButton > button {
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important;
        color: white !important;
        border: none !important;
        border-radius: 12px !important;
        padding: 12px 24px !important;
        font-weight: 600 !important;
        width: 100%;
        transition: all 0.2s ease !important;
        box-shadow: 0 4px 15px rgba(0,0,0,0.2);
    }

    .stButton > button:hover {
        transform: translateY(-2px);
        box-shadow: 0 10px 25px rgba(102, 126, 234, 0.4);
    }

    /* Secondary Button */
    .secondary-btn .stButton > button {
        background: rgba(255, 255, 255, 0.15) !important;
        color: white !important;
        border: 2px solid rgba(255,255,255,0.3) !important;
        backdrop-filter: blur(5px);
    }

    /* Checkbox */
    .stCheckbox {
        margin: 1rem 0 !important;
        color: white !important;
    }

    .stCheckbox label {
        color: white !important;
    }

    /* Forgot password link */
    .forgot-link .stButton > button {
        background: transparent !important;
        color: white !important;
        border: none !important;
        padding: 0 !important;
        font-size: 0.9rem !important;
        text-decoration: underline;
        box-shadow: none !important;
    }

    .forgot-link .stButton > button:hover {
        color: rgba(255,255,255,0.8) !important;
        transform: none !important;
    }

    /* Dashboard Welcome */
    .welcome-header {
        color: white !important;
        font-size: 2.8rem !important;
        font-weight: 700 !important;
        margin-bottom: 0.5rem !important;
        text-shadow: 0 2px 10px rgba(0,0,0,0.3);
        background: linear-gradient(135deg, rgba(255,255,255,0.2), rgba(255,255,255,0.1));
        padding: 0.8rem 2rem;
        border-radius: 50px;
        display: inline-block;
        backdrop-filter: blur(5px);
        border: 1px solid rgba(255,255,255,0.2);
    }

    .welcome-subheader {
        color: white !important;
        font-size: 1.5rem !important;
        font-weight: 400 !important;
        margin-bottom: 2rem !important;
        text-shadow: 0 1px 5px rgba(0,0,0,0.2);
        background: rgba(0,0,0,0.2);
        padding: 0.5rem 2rem;
        border-radius: 40px;
        display: inline-block;
    }

    /* Chat Styles */
    .user-msg {
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        color: white;
        padding: 12px 18px;
        border-radius: 20px 20px 4px 20px;
        margin: 10px 0;
        max-width: 70%;
        margin-left: auto;
        box-shadow: 0 2px 10px rgba(0,0,0,0.1);
        display: inline-block;
        float: right;
        clear: both;
    }

    .bot-msg {
        background: rgba(255, 255, 255, 0.95);
        color: #1a1f36;
        padding: 12px 18px;
        border-radius: 20px 20px 20px 4px;
        margin: 10px 0;
        max-width: 70%;
        margin-right: auto;
        box-shadow: 0 2px 10px rgba(0,0,0,0.1);
        display: inline-block;
        float: left;
        clear: both;
        border: 1px solid rgba(102, 126, 234, 0.2);
    }

    .chat-container {
        width: 100%;
        min-height: 350px;
        margin: 20px 0;
        overflow-y: auto;
        background: rgba(255,255,255,0.05);
        border-radius: 20px;
        padding: 1.5rem;
        border: 1px solid rgba(255,255,255,0.1);
    }

    .chat-container:after {
        content: "";
        display: table;
        clear: both;
    }

    /* Quick action buttons */
    .quick-action-btn .stButton > button {
        background: rgba(255, 255, 255, 0.15) !important;
        color: white !important;
        border: 1px solid rgba(255,255,255,0.3) !important;
        border-radius: 25px !important;
        padding: 8px 16px !important;
        font-size: 0.9rem !important;
        backdrop-filter: blur(5px);
    }

    /* Readability Metrics */
    .metric-card {
        background: rgba(255,255,255,0.1);
        backdrop-filter: blur(10px);
        border-radius: 15px;
        padding: 20px;
        border: 1px solid rgba(255,255,255,0.2);
        margin: 10px 0;
    }

    .metric-value {
        font-size: 2.5rem;
        font-weight: 700;
        color: white;
        text-align: center;
        text-shadow: 0 2px 10px rgba(102, 126, 234, 0.5);
    }

    .metric-label {
        color: rgba(255,255,255,0.8);
        text-align: center;
        font-size: 1rem;
        margin-top: 5px;
    }

    /* Level Badge */
    .level-badge {
        display: inline-block;
        padding: 8px 20px;
        border-radius: 30px;
        font-weight: 600;
        font-size: 1.2rem;
        margin: 10px 0;
    }

    .level-beginner { background: linear-gradient(135deg, #28a745, #20c997); color: white; }
    .level-intermediate { background: linear-gradient(135deg, #17a2b8, #0dcaf0); color: white; }
    .level-advanced { background: linear-gradient(135deg, #ffc107, #fd7e14); color: #1a1f36; }
    .level-expert { background: linear-gradient(135deg, #dc3545, #c82333); color: white; }

    /* Option Cards */
    .option-card {
        background: rgba(255, 255, 255, 0.1);
        backdrop-filter: blur(10px);
        border: 2px solid rgba(255,255,255,0.2);
        border-radius: 15px;
        padding: 20px;
        margin: 10px 0;
        cursor: pointer;
        transition: all 0.3s ease;
        text-align: center;
    }

    .option-card:hover {
        background: rgba(255, 255, 255, 0.2);
        border-color: #667eea;
        transform: translateY(-5px);
    }

    .option-icon {
        font-size: 3rem;
        margin-bottom: 10px;
    }

    .option-title {
        color: white;
        font-size: 1.3rem;
        font-weight: 600;
        margin-bottom: 5px;
    }

    .option-desc {
        color: rgba(255,255,255,0.7);
        font-size: 0.9rem;
    }

    /* Footer */
    .footer {
        text-align: center;
        color: rgba(255,255,255,0.9);
        padding: 2rem 0 1rem 0;
        font-size: 0.9rem;
    }

    /* Hide Streamlit branding */
    #MainMenu {visibility: hidden;}
    footer {visibility: hidden;}
</style>
""", unsafe_allow_html=True)

# ================= READABILITY HELPER FUNCTIONS =================
def create_gauge(value, title, min_val=0, max_val=100, color="#667eea"):
    """Create a gauge chart for readability metrics"""
    fig = go.Figure(go.Indicator(
        mode="gauge+number",
        value=value,
        title={'text': title, 'font': {'color': 'white', 'size': 14}},
        number={'font': {'color': 'white', 'size': 20}},
        gauge={
            'axis': {'range': [min_val, max_val], 'tickwidth': 1, 'tickcolor': 'white'},
            'bar': {'color': color},
            'bgcolor': "rgba(255,255,255,0.1)",
            'borderwidth': 2,
            'bordercolor': "rgba(255,255,255,0.2)",
            'steps': [
                {'range': [min_val, max_val], 'color': "rgba(255,255,255,0.05)"}
            ],
        }
    ))
    fig.update_layout(
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        font={'color': "white", 'family': "Inter"},
        height=200,
        margin=dict(l=10, r=10, t=40, b=10)
    )
    return fig

def analyze_readability(text):
    """Analyze text readability and return metrics"""
    analyzer = readability.ReadabilityAnalyzer(text)
    return analyzer.get_all_metrics()

# ================= SIGNUP =================
def signup():
    col1, col2, col3 = st.columns([1, 2.2, 1])
    with col2:
        st.markdown("<h1 class='project-title'>PolicyNav Pro</h1>", unsafe_allow_html=True)
        st.markdown("<p class='project-subtitle'>Create Your Account</p>", unsafe_allow_html=True)

        username = st.text_input("Username", placeholder="Choose a username", key="signup_username")
        email = st.text_input("Email", placeholder="Enter your email", key="signup_email")

        col_a, col_b = st.columns(2)
        with col_a:
            password = st.text_input("Password", type="password", placeholder="Create password", key="signup_pass")
        with col_b:
            confirm_password = st.text_input("Confirm", type="password", placeholder="Confirm password", key="signup_confirm")

        security_question = st.selectbox("Security Question", [
            "What is your pet's name?",
            "What is your mother's maiden name?",
            "What was your first car?",
            "What city were you born in?"
        ])

        security_answer = st.text_input("Security Answer", placeholder="Your answer", key="signup_answer")

        terms = st.checkbox("I agree to the Terms of Service")

        if st.button("Create Account", key="signup_btn"):
            if not username or not email or not password or not confirm_password or not security_answer:
                st.error("❌ All fields are required!")
            elif not re.match(r"[^@]+@[^@]+\.[a-zA-Z]{2,}", email):
                st.error("❌ Please enter a valid email address!")
            elif len(password) < 8:
                st.error("❌ Password must be at least 8 characters long!")
            elif password != confirm_password:
                st.error("❌ Passwords do not match!")
            elif not terms:
                st.error("❌ Please accept the terms!")
            else:
                with st.spinner("Creating account..."):
                    success, message = save_user(username, email, password, security_question, security_answer)
                    if success:
                        st.session_state.jwt = create_token(email, username)
                        update_login_stats(email)
                        st.success("✅ Account created successfully!")
                        st.balloons()
                        time.sleep(1.5)
                        st.rerun()
                    else:
                        st.error(f"❌ {message}")

        st.markdown('<div class="secondary-btn">', unsafe_allow_html=True)
        if st.button("← Back to Login", key="back_to_login"):
            st.session_state.page = "login"
            st.rerun()
        st.markdown('</div>', unsafe_allow_html=True)

        st.markdown('</div>', unsafe_allow_html=True)

# ================= LOGIN =================
def login():
    col1, col2, col3 = st.columns([1, 2.2, 1])
    with col2:
        # Enhanced Project Title
        st.markdown("<h1 class='project-title' >PolicyNav Login</h1>", unsafe_allow_html=True)
        st.markdown("<p class='auth-subtitle'>Sign in to continue your journey</p>", unsafe_allow_html=True)

        email = st.text_input("Email", placeholder="Enter your email", key="login_email")
        password = st.text_input("Password", type="password", placeholder="Enter your password", key="login_password")

        col_rem, col_forgot = st.columns(2)
        with col_rem:
            remember = st.checkbox("Remember me")
        with col_forgot:
            st.markdown('<div class="forgot-link">', unsafe_allow_html=True)
            if st.button("Forgot Password?", key="forgot_link"):
                st.session_state.page = "forgot"
                st.session_state.reset_method = None
                st.session_state.otp_sent = False
                st.session_state.otp_verified = False
                st.rerun()
            st.markdown('</div>', unsafe_allow_html=True)

        if st.button("Sign In", key="login_btn"):
            if not email or not password:
                st.error("❌ Please enter both email and password!")
            else:
                with st.spinner("Verifying..."):
                    user = get_user(email)
                    if user and user[1] == hash_password(password):
                        st.session_state.jwt = create_token(email, user[0])
                        update_login_stats(email)
                        st.success("✅ Login successful!")
                        st.balloons()
                        time.sleep(1.5)
                        st.rerun()
                    else:
                        st.error("❌ Invalid email or password!")

        st.markdown("<br>", unsafe_allow_html=True)
        st.markdown("<p style='text-align: center; color: white; font-size: 1rem;'>Don't have an account</p>", unsafe_allow_html=True)
        st.markdown('<div class="secondary-btn">', unsafe_allow_html=True)
        if st.button("Create New Account", key="create_btn"):
            st.session_state.page = "signup"
            st.rerun()
        st.markdown('</div>', unsafe_allow_html=True)

        st.markdown('</div>', unsafe_allow_html=True)

# ================= FORGOT PASSWORD - METHOD SELECTION =================
def forgot_method_selection():
    col1, col2, col3 = st.columns([1, 2.2, 1])
    with col2:
        st.markdown("<h1 class='project-title'>PolicyNav Pro</h1>", unsafe_allow_html=True)
        st.markdown("<h2 class='auth-header'>Reset Password</h2>", unsafe_allow_html=True)
        st.markdown("<p class='auth-subtitle'>Choose how you want to reset your password</p>", unsafe_allow_html=True)

        # Two options in cards
        col_a, col_b = st.columns(2)

        with col_a:
            st.markdown("""
            <div class="option-card" onclick="document.getElementById('security_btn').click()">
                <div class="option-icon">🔐</div>
                <div class="option-title">Security Question</div>
                <div class="option-desc">Answer your security question to reset password</div>
            </div>
            """, unsafe_allow_html=True)
            if st.button("Select Security Question", key="security_btn", use_container_width=True):
                st.session_state.reset_method = "security"
                st.rerun()

        with col_b:
            st.markdown("""
            <div class="option-card" onclick="document.getElementById('otp_btn').click()">
                <div class="option-icon">📧</div>
                <div class="option-title">Email OTP</div>
                <div class="option-desc">Receive OTP on your email to reset password</div>
            </div>
            """, unsafe_allow_html=True)
            if st.button("Select Email OTP", key="otp_btn", use_container_width=True):
                st.session_state.reset_method = "otp"
                st.rerun()

        st.markdown("<br>", unsafe_allow_html=True)
        st.markdown('<div class="secondary-btn">', unsafe_allow_html=True)
        if st.button("← Back to Login", key="back_to_login_from_forgot"):
            st.session_state.page = "login"
            st.session_state.reset_method = None
            st.rerun()
        st.markdown('</div>', unsafe_allow_html=True)

# ================= FORGOT PASSWORD - SECURITY QUESTION METHOD =================
def forgot_security():
    col1, col2, col3 = st.columns([1, 2.2, 1])
    with col2:
        st.markdown("<h1 class='project-title'>PolicyNav Pro</h1>", unsafe_allow_html=True)
        st.markdown("<h2 class='auth-header'>Security Question</h2>", unsafe_allow_html=True)
        st.markdown("<p class='auth-subtitle'>Answer your security question</p>", unsafe_allow_html=True)

        if "q" not in st.session_state or not st.session_state.q:
            email = st.text_input("Registered Email", placeholder="Enter your email", key="reset_email_security")

            if st.button("Verify Email", key="get_question_btn"):
                if not email:
                    st.error("❌ Please enter your email!")
                else:
                    with st.spinner("Searching..."):
                        if check_user_exists(email):
                            question = get_question(email)
                            if question:
                                st.session_state.reset = email
                                st.session_state.q = question
                                st.success("✅ Email verified! Answer security question.")
                                st.rerun()
                            else:
                                st.error("❌ Error fetching security question!")
                        else:
                            st.error("❌ Email not found!")

        if "q" in st.session_state and st.session_state.q:
            st.info(f"**Security Question:** {st.session_state.q}")

            answer = st.text_input("Your Answer", placeholder="Enter your answer", key="security_answer_input")
            new_password = st.text_input("New Password", type="password", placeholder="Enter new password", key="new_password_security")
            confirm_new = st.text_input("Confirm Password", type="password", placeholder="Confirm password", key="confirm_password_security")

            col_a, col_b = st.columns(2)
            with col_a:
                if st.button("Reset Password", key="reset_security_btn"):
                    if not answer or not new_password:
                        st.error("❌ Please fill all fields!")
                    elif len(new_password) < 8:
                        st.error("❌ Password must be at least 8 characters!")
                    elif new_password != confirm_new:
                        st.error("❌ Passwords don't match!")
                    else:
                        with st.spinner("Updating..."):
                            if check_answer(st.session_state.reset, answer):
                                update_password(st.session_state.reset, new_password)
                                st.success("✅ Password updated successfully!")
                                st.balloons()
                                st.session_state.q = None
                                st.session_state.reset = None
                                st.session_state.reset_method = None
                                time.sleep(1.5)
                                st.session_state.page = "login"
                                st.rerun()
                            else:
                                st.error("❌ Incorrect answer!")

            with col_b:
                if st.button("← Change Method", key="back_to_methods"):
                    st.session_state.q = None
                    st.session_state.reset = None
                    st.session_state.reset_method = None
                    st.rerun()

# ================= FORGOT PASSWORD - OTP METHOD =================
def forgot_otp():
    col1, col2, col3 = st.columns([1, 2.2, 1])
    with col2:
        st.markdown("<h1 class='project-title'>PolicyNav Pro</h1>", unsafe_allow_html=True)
        st.markdown("<h2 class='auth-header'>Email OTP Verification</h2>", unsafe_allow_html=True)
        st.markdown("<p class='auth-subtitle'>Verify your identity via OTP</p>", unsafe_allow_html=True)

        # Step 1: Email input and send OTP
        if not st.session_state.otp_sent:
            email = st.text_input("Registered Email", placeholder="Enter your email", key="reset_email_otp")

            if st.button("Send OTP", key="send_otp_btn"):
                if not email:
                    st.error("❌ Please enter your email!")
                else:
                    with st.spinner("Checking email..."):
                        if check_user_exists(email):
                            # Generate and save OTP
                            otp = generate_otp()
                            if save_otp(email, otp):
                                # Send OTP via email
                                if send_otp_email(email, otp):
                                    st.session_state.reset = email
                                    st.session_state.otp_sent = True
                                    st.success("✅ OTP sent to your email! Please check your inbox.")
                                    st.rerun()
                                else:
                                    st.error("❌ Failed to send OTP. Please try again.")
                            else:
                                st.error("❌ Error generating OTP. Please try again.")
                        else:
                            st.error("❌ Email not found!")

        # Step 2: OTP verification
        elif st.session_state.otp_sent and not st.session_state.otp_verified:
            st.info(f"📧 OTP sent to: {st.session_state.reset}")

            otp_input = st.text_input("Enter OTP", placeholder="Enter 6-digit OTP", key="otp_input", max_chars=6)

            col_a, col_b, col_c = st.columns(3)
            with col_a:
                if st.button("Verify OTP", key="verify_otp_btn"):
                    if not otp_input:
                        st.error("❌ Please enter OTP!")
                    elif len(otp_input) != 6 or not otp_input.isdigit():
                        st.error("❌ Please enter a valid 6-digit OTP!")
                    else:
                        with st.spinner("Verifying OTP..."):
                            if verify_otp(st.session_state.reset, otp_input):
                                st.session_state.otp_verified = True
                                st.success("✅ OTP verified! Please set your new password.")
                                st.rerun()
                            else:
                                st.error("❌ Invalid or expired OTP!")

            with col_b:
                if st.button("Resend OTP", key="resend_otp_btn"):
                    with st.spinner("Sending new OTP..."):
                        otp = generate_otp()
                        if save_otp(st.session_state.reset, otp) and send_otp_email(st.session_state.reset, otp):
                            st.success("✅ New OTP sent!")
                            st.rerun()
                        else:
                            st.error("❌ Failed to send OTP. Please try again.")

            with col_c:
                if st.button("← Change Email", key="change_email_btn"):
                    st.session_state.otp_sent = False
                    st.rerun()

        # Step 3: Set new password
        elif st.session_state.otp_verified:
            st.success("✅ OTP verified successfully!")

            new_password = st.text_input("New Password", type="password", placeholder="Enter new password", key="new_password_otp")
            confirm_new = st.text_input("Confirm Password", type="password", placeholder="Confirm password", key="confirm_password_otp")

            col_a, col_b = st.columns(2)
            with col_a:
                if st.button("Reset Password", key="reset_otp_btn"):
                    if not new_password:
                        st.error("❌ Please enter new password!")
                    elif len(new_password) < 8:
                        st.error("❌ Password must be at least 8 characters!")
                    elif new_password != confirm_new:
                        st.error("❌ Passwords don't match!")
                    else:
                        with st.spinner("Updating password..."):
                            update_password(st.session_state.reset, new_password)
                            st.success("✅ Password updated successfully!")
                            st.balloons()

                            # Reset session state
                            st.session_state.reset = None
                            st.session_state.otp_sent = False
                            st.session_state.otp_verified = False
                            st.session_state.reset_method = None

                            time.sleep(1.5)
                            st.session_state.page = "login"
                            st.rerun()

            with col_b:
                if st.button("← Change Method", key="back_to_methods_otp"):
                    st.session_state.reset = None
                    st.session_state.otp_sent = False
                    st.session_state.otp_verified = False
                    st.session_state.reset_method = None
                    st.rerun()

# ================= FORGOT PASSWORD MAIN =================
def forgot():
    if st.session_state.reset_method is None:
        forgot_method_selection()
    elif st.session_state.reset_method == "security":
        forgot_security()
    elif st.session_state.reset_method == "otp":
        forgot_otp()

# ================= CHAT DASHBOARD =================
def chat_dashboard():
    data = verify_token(st.session_state.jwt)
    username = data.get("username", "User")
    email = data.get("sub", "")

    st.markdown(f"<h1 class='welcome-header'>Welcome back, {username}! 👋</h1>", unsafe_allow_html=True)
    st.markdown("<p class='welcome-subheader'>How can I assist you with policies today?</p>", unsafe_allow_html=True)

    # Chat container
    if 'chat_history' not in st.session_state:
        st.session_state.chat_history = [{
            'type': 'bot',
            'message': 'Hello! I am PolicyNav Pro. Ask me anything about policies, regulations, or compliance!'
        }]

    st.markdown('<div class="chat-container">', unsafe_allow_html=True)
    for msg in st.session_state.chat_history:
        if msg['type'] == 'user':
            st.markdown(f'<div class="user-msg">{msg["message"]}</div>', unsafe_allow_html=True)
        else:
            st.markdown(f'<div class="bot-msg">{msg["message"]}</div>', unsafe_allow_html=True)

    st.markdown('</div>', unsafe_allow_html=True)

    # User input
    st.markdown('<div style="margin: 20px 0;">', unsafe_allow_html=True)

    col1, col2 = st.columns([6, 1])

    with col1:
        input_key = f"chat_input_{st.session_state.chat_input_key}"
        user_input = st.text_input(
            "Message",
            placeholder="Ask about policies, regulations, or compliance...",
            label_visibility="collapsed",
            key=input_key
        )

    with col2:
        send_button = st.button("📤 Send", use_container_width=True)

    if send_button and user_input:
        st.session_state.chat_history.append({
            'type': 'user',
            'message': user_input
        })

        # Simulated response
        response = f"I understand you're asking about: '{user_input}'. As PolicyNav Pro, I can help you navigate through complex policies and regulations. Please provide more details so I can assist you better."

        st.session_state.chat_history.append({
            'type': 'bot',
            'message': response
        })

        st.session_state.chat_input_key += 1
        st.rerun()

    st.markdown('</div>', unsafe_allow_html=True)

    # Quick action buttons
    st.markdown("<br>", unsafe_allow_html=True)
    col1, col2, col3, col4 = st.columns(4)

    with col1:
        if st.button("📊 Policy Analysis", use_container_width=True):
            st.session_state.chat_history.append({
                'type': 'user',
                'message': "Can you help me analyze this policy?"
            })
            st.rerun()

    with col2:
        if st.button("⚖️ Compliance Check", use_container_width=True):
            st.session_state.chat_history.append({
                'type': 'user',
                'message': "Check compliance for my business"
            })
            st.rerun()

    with col3:
        if st.button("📝 Document Review", use_container_width=True):
            st.session_state.chat_history.append({
                'type': 'user',
                'message': "Review this policy document"
            })
            st.rerun()

    with col4:
        if st.button("🔍 Regulation Search", use_container_width=True):
            st.session_state.chat_history.append({
                'type': 'user',
                'message': "Find regulations for my industry"
            })
            st.rerun()

# ================= READABILITY ANALYZER PAGE =================
def readability_analyzer():
    data = verify_token(st.session_state.jwt)
    username = data.get("username", "User")

    st.markdown(f"<h1 class='welcome-header'>📖 Readability Analyzer</h1>", unsafe_allow_html=True)
    st.markdown("<p class='welcome-subheader'>Analyze and improve your document readability</p>", unsafe_allow_html=True)

    # Input Method
    tab1, tab2 = st.tabs(["✍️ Input Text", "📂 Upload File"])
    text_input = ""

    with tab1:
        text_input = st.text_area(
            "Enter text to analyze (minimum 50 characters):",
            height=200,
            placeholder="Paste your policy document, article, or any text here..."
        )

    with tab2:
        uploaded_file = st.file_uploader("Upload a file", type=["txt", "pdf"])
        if uploaded_file:
            try:
                if uploaded_file.type == "application/pdf":
                    reader = PyPDF2.PdfReader(uploaded_file)
                    text = ""
                    for page in reader.pages:
                        text += page.extract_text() + "\n"
                    text_input = text
                    st.success(f"✅ Loaded {len(reader.pages)} pages from PDF.")
                else:
                    text_input = uploaded_file.read().decode("utf-8")
                    st.success(f"✅ Loaded TXT file: {uploaded_file.name}")
            except Exception as e:
                st.error(f"Error reading file: {e}")

    # Analyze Button
    if st.button("🔍 Analyze Readability", type="primary", use_container_width=True):
        if len(text_input) < 50:
            st.error("❌ Text is too short. Please enter at least 50 characters.")
        else:
            with st.spinner("Calculating readability metrics..."):
                try:
                    score = analyze_readability(text_input)

                    # Calculate average grade level
                    avg_grade = (score['Flesch-Kincaid Grade'] + score['Gunning Fog'] +
                                score['SMOG Index'] + score['Coleman-Liau']) / 4

                    # Determine reading level
                    if avg_grade <= 6:
                        level, level_class = "Beginner (Elementary)", "level-beginner"
                    elif avg_grade <= 10:
                        level, level_class = "Intermediate (Middle School)", "level-intermediate"
                    elif avg_grade <= 14:
                        level, level_class = "Advanced (High School/College)", "level-advanced"
                    else:
                        level, level_class = "Expert (Professional/Academic)", "level-expert"

                    # Display Level Badge
                    st.markdown(f"""
                    <div style="text-align: center; margin: 20px 0;">
                        <div class="level-badge {level_class}">
                            📊 Reading Level: {level} (Grade {avg_grade:.1f})
                        </div>
                    </div>
                    """, unsafe_allow_html=True)

                    # Metrics Row 1
                    col1, col2, col3, col4, col5 = st.columns(5)
                    with col1:
                        st.markdown('<div class="metric-card">', unsafe_allow_html=True)
                        st.markdown(f'<div class="metric-value">{score["Flesch Reading Ease"]:.1f}</div>', unsafe_allow_html=True)
                        st.markdown('<div class="metric-label">Flesch Ease</div>', unsafe_allow_html=True)
                        st.markdown('</div>', unsafe_allow_html=True)

                    with col2:
                        st.markdown('<div class="metric-card">', unsafe_allow_html=True)
                        st.markdown(f'<div class="metric-value">{score["Flesch-Kincaid Grade"]:.1f}</div>', unsafe_allow_html=True)
                        st.markdown('<div class="metric-label">Flesch-Kincaid</div>', unsafe_allow_html=True)
                        st.markdown('</div>', unsafe_allow_html=True)

                    with col3:
                        st.markdown('<div class="metric-card">', unsafe_allow_html=True)
                        st.markdown(f'<div class="metric-value">{score["Gunning Fog"]:.1f}</div>', unsafe_allow_html=True)
                        st.markdown('<div class="metric-label">Gunning Fog</div>', unsafe_allow_html=True)
                        st.markdown('</div>', unsafe_allow_html=True)

                    with col4:
                        st.markdown('<div class="metric-card">', unsafe_allow_html=True)
                        st.markdown(f'<div class="metric-value">{score["SMOG Index"]:.1f}</div>', unsafe_allow_html=True)
                        st.markdown('<div class="metric-label">SMOG Index</div>', unsafe_allow_html=True)
                        st.markdown('</div>', unsafe_allow_html=True)

                    with col5:
                        st.markdown('<div class="metric-card">', unsafe_allow_html=True)
                        st.markdown(f'<div class="metric-value">{score["Coleman-Liau"]:.1f}</div>', unsafe_allow_html=True)
                        st.markdown('<div class="metric-label">Coleman-Liau</div>', unsafe_allow_html=True)
                        st.markdown('</div>', unsafe_allow_html=True)

                    # Gauges
                    st.markdown("### 📈 Detailed Metrics Visualization")
                    col1, col2, col3 = st.columns(3)

                    with col1:
                        st.plotly_chart(
                            create_gauge(score["Flesch Reading Ease"], "Flesch Reading Ease", 0, 100, "#667eea"),
                            use_container_width=True
                        )

                    with col2:
                        st.plotly_chart(
                            create_gauge(score["Flesch-Kincaid Grade"], "Flesch-Kincaid Grade", 0, 20, "#764ba2"),
                            use_container_width=True
                        )

                    with col3:
                        st.plotly_chart(
                            create_gauge(score["SMOG Index"], "SMOG Index", 0, 20, "#ff6b6b"),
                            use_container_width=True
                        )

                    col4, col5 = st.columns(2)

                    with col4:
                        st.plotly_chart(
                            create_gauge(score["Gunning Fog"], "Gunning Fog", 0, 20, "#4ecdc4"),
                            use_container_width=True
                        )

                    with col5:
                        st.plotly_chart(
                            create_gauge(score["Coleman-Liau"], "Coleman-Liau", 0, 20, "#45b7d1"),
                            use_container_width=True
                        )

                    # Text Statistics
                    st.markdown("### 📊 Text Statistics")
                    col1, col2, col3, col4, col5 = st.columns(5)

                    analyzer = readability.ReadabilityAnalyzer(text_input)

                    with col1:
                        st.metric("Sentences", analyzer.num_sentences)
                    with col2:
                        st.metric("Words", analyzer.num_words)
                    with col3:
                        st.metric("Syllables", analyzer.num_syllables)
                    with col4:
                        st.metric("Complex Words", analyzer.complex_words)
                    with col5:
                        st.metric("Characters", analyzer.char_count)

                except Exception as e:
                    st.error(f"Error analyzing text: {str(e)}")

# ================= DASHBOARD WITH TABBED INTERFACE =================
def dashboard():
    data = verify_token(st.session_state.jwt)
    if not data:
        st.session_state.jwt = None
        st.session_state.page = "login"
        st.rerun()
        return

    username = data.get("username", "User")
    email = data.get("sub", "")

    # Sidebar
    with st.sidebar:
        st.markdown(f"""
        <div style='text-align: center; padding: 1rem 0;'>
            <div class="sidebar-avatar">
                {username[0].upper()}
            </div>
            <div style='color: white; font-size: 1.3rem; font-weight: 600;'>{username}</div>
            <div style='color: rgba(255,255,255,0.7); font-size: 0.85rem; word-break: break-all;'>{email}</div>
        </div>
        """, unsafe_allow_html=True)

        st.markdown("---")

        # Navigation Options
        if st.button("💬 Chat", use_container_width=True,
                    type="primary" if st.session_state.selected_tab == "Chat" else "secondary"):
            st.session_state.selected_tab = "Chat"
            st.rerun()

        if st.button("📊 Readability Analyzer", use_container_width=True,
                    type="primary" if st.session_state.selected_tab == "Readability" else "secondary"):
            st.session_state.selected_tab = "Readability"
            st.rerun()

        st.markdown("---")
        st.markdown('<p style="color: rgba(255,255,255,0.6); font-size: 0.85rem;">Recent Chats</p>', unsafe_allow_html=True)

        history_items = [
            "Healthcare policy analysis",
            "Insurance claim process",
            "Tax regulation 2024",
            "Compliance checklist"
        ]

        for item in history_items:
            if st.button(f"📄 {item}", key=f"history_{item}", use_container_width=True):
                if 'chat_history' in st.session_state:
                    st.session_state.chat_history.append({
                        'type': 'user',
                        'message': f"Tell me about {item}"
                    })
                    st.session_state.selected_tab = "Chat"
                    st.rerun()

        st.markdown("---")

        if st.button("🚪 Logout", use_container_width=True):
            st.session_state.jwt = None
            st.session_state.page = "login"
            st.rerun()

    # Main Content based on selected tab
    if st.session_state.selected_tab == "Chat":
        chat_dashboard()
    else:
        readability_analyzer()

# ================= FOOTER =================
def footer():
    st.markdown("""
    <div class="footer">
        <p>© 2024 PolicyNav Pro - Your Intelligent Policy Navigation Platform</p>
    </div>
    """, unsafe_allow_html=True)

# ================= MAIN =================
try:
    if st.session_state.jwt:
        dashboard()
    else:
        if st.session_state.page == "signup":
            signup()
        elif st.session_state.page == "forgot":
            forgot()
        else:
            login()

    footer()

except Exception as e:
    st.error(f"Error: {str(e)}")

Writing app.py


In [ ]:
# ================= POLICYNAV PRO - NGROK LAUNCHER (FIXED VERSION) =================
import os
import subprocess
import time
import socket
import sys
import requests
from pyngrok import ngrok, conf

# ================= CONFIGURATION =================
PORT = 8501
JWT_SECRET = "super-secret-change-me"

# ================= FIX NGROK CONFIG =================
def setup_ngrok(token):
    """Setup ngrok with proper configuration"""
    try:
        # Set ngrok config path
        conf.get_default().config_path = None  # Use default config

        # Try to set auth token with error handling
        try:
            ngrok.set_auth_token(token)
        except Exception as e:
            print(f"⚠️ Ngrok set_auth_token error: {e}")
            print("Trying alternative method...")

            # Alternative: Direct config file creation
            import json
            import os
            from pathlib import Path

            ngrok_config_path = Path.home() / '.config' / 'ngrok' / 'ngrok.yml'
            ngrok_config_path.parent.mkdir(parents=True, exist_ok=True)

            config = {
                "version": "2",
                "authtoken": token
            }

            with open(ngrok_config_path, 'w') as f:
                json.dump(config, f)

            print(f"✅ Ngrok config created at {ngrok_config_path}")

        return True
    except Exception as e:
        print(f"❌ Ngrok setup error: {e}")
        return False

# ================= WAIT FOR STREAMLIT =================
def wait_for_streamlit(port=PORT, timeout=30):
    """Wait for Streamlit to start"""
    print("⏳ Waiting for Streamlit to start...", end="", flush=True)
    start_time = time.time()
    while time.time() - start_time < timeout:
        try:
            sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            sock.settimeout(1)
            result = sock.connect_ex(('localhost', port))
            sock.close()
            if result == 0:
                print(" ✅")
                return True
        except:
            pass
        print(".", end="", flush=True)
        time.sleep(1)
    print(" ❌")
    return False

# ================= CLEANUP =================
def cleanup():
    """Clean up processes"""
    print("\n🧹 Cleaning up...")
    try:
        if 'process' in locals():
            process.terminate()
            process.wait(timeout=5)

        # Kill ngrok processes
        try:
            ngrok.kill()
        except:
            pass

        # Force kill any remaining ngrok processes
        os.system("pkill -f ngrok >/dev/null 2>&1")
        os.system("pkill -f streamlit >/dev/null 2>&1")
        print("✅ Cleanup complete")
    except Exception as e:
        print(f"⚠️ Cleanup warning: {e}")

# ================= MAIN LAUNCHER =================
def main():
    print("=" * 60)
    print("🚀 POLICYNAV PRO - NGROK LAUNCHER (FIXED VERSION)")
    print("=" * 60)

    print("\n📌 Step 1: Ngrok Authentication")
    print("📌 Get your token from: https://dashboard.ngrok.com/get-started/your-authtoken")
    ngrok_token = input("🔑 Enter Ngrok Authtoken: ").strip()

    if not ngrok_token:
        print("❌ Ngrok token is required!")
        return

    # Step 2: Email Password (optional)
    print("\n📧 Step 2: Email Configuration (Optional)")
    print("📧 For OTP functionality, enter Gmail App Password")
    print("📧 Get it from: https://myaccount.google.com/apppasswords")
    email_app_password = input("🔐 Enter Email App Password (press Enter to skip): ").strip()

    try:
        # Setup Ngrok with fixed configuration
        print("\n🔧 Configuring Ngrok...")
        if not setup_ngrok(ngrok_token):
            print("❌ Failed to configure ngrok")
            return

        # Kill existing processes
        print("🔄 Stopping existing processes...")
        os.system("pkill -f streamlit >/dev/null 2>&1")
        time.sleep(2)

        # Kill any existing ngrok processes
        try:
            ngrok.kill()
        except:
            pass
        os.system("pkill -f ngrok >/dev/null 2>&1")
        time.sleep(2)

        # Setup environment
        env = os.environ.copy()
        env['JWT_SECRET'] = JWT_SECRET
        if email_app_password:
            env['EMAIL_APP_PASSWORD'] = email_app_password
            print("✅ Email OTP configured")

        # Start Streamlit
        print("\n🚀 Starting Streamlit server...")
        process = subprocess.Popen(
            ["streamlit", "run", "app.py", "--server.port", str(PORT), "--server.headless", "true"],
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE
        )

        # Wait for Streamlit
        if not wait_for_streamlit():
            print("❌ Streamlit failed to start!")
            return

        # Create Ngrok tunnel with retry
        print("🔗 Creating Ngrok tunnel...")
        max_retries = 3
        public_url = None

        for attempt in range(max_retries):
            try:
                # Try different connection methods
                try:
                    # Method 1: Standard connect
                    tunnel = ngrok.connect(PORT, "http")
                    public_url = tunnel.public_url
                    break
                except:
                    # Method 2: With protocol specified
                    tunnel = ngrok.connect(PORT, proto="http")
                    public_url = tunnel.public_url
                    break
            except Exception as e:
                print(f"⚠️ Tunnel attempt {attempt + 1} failed: {e}")
                if attempt < max_retries - 1:
                    print("Retrying...")
                    time.sleep(3)
                    ngrok.kill()
                    time.sleep(2)
                else:
                    print("❌ Failed to create ngrok tunnel after multiple attempts")
                    print("\nTrying alternative: Using local URL only...")
                    public_url = f"http://localhost:{PORT}"

        # Display success
        print("\n" + "=" * 60)
        print("✅ APPLICATION STATUS")
        print("=" * 60)
        print(f"\n🌐 PUBLIC URL: {public_url if public_url else 'Not available'}")
        print(f"📱 LOCAL URL: http://localhost:{PORT}")

        if "ngrok" not in str(public_url):
            print("\n⚠️ Ngrok tunnel failed. App is running locally only.")
            print("💡 Tips to fix ngrok:")
            print("   1. Check your internet connection")
            print("   2. Verify ngrok token is correct")
            print("   3. Try: pip install --upgrade pyngrok")
            print("   4. Restart the kernel and run again")

        print("\n" + "=" * 60)

        if not email_app_password:
            print("\n⚠️ Note: OTP features are disabled (no email password)")
        else:
            print("\n✅ OTP features are enabled")

        # Interactive mode
        print("\n" + "-" * 40)
        print("COMMANDS:")
        print("  q - Quit and stop server")
        print("  s - Show status")
        print("-" * 40)

        while True:
            try:
                cmd = input("\nEnter command: ").strip().lower()

                if cmd == 'q':
                    print("\n🛑 Stopping server...")
                    break

                elif cmd == 's':
                    # Check Streamlit status
                    if process.poll() is None:
                        print("✅ Streamlit: Running")
                    else:
                        print("❌ Streamlit: Stopped")

                    # Check ngrok status
                    try:
                        tunnels = ngrok.get_tunnels()
                        if tunnels:
                            print(f"✅ Ngrok: Active - {tunnels[0].public_url}")
                        else:
                            print("❌ Ngrok: No active tunnel")
                    except:
                        print("❌ Ngrok: Not responding")

                else:
                    print("❌ Unknown command. Use: q, s")

            except KeyboardInterrupt:
                print("\n\n🛑 Stopping server...")
                break

    except Exception as e:
        print(f"\n❌ Error: {e}")
        import traceback
        traceback.print_exc()

    finally:
        cleanup()
        print("\n👋 Thank you for using PolicyNav Pro!")

# ================= QUICK LAUNCH WITH ENV VARS =================
def quick_launch(ngrok_token, email_password=None):
    """Quick launch with direct parameters"""
    process = None
    try:
        # Setup ngrok
        setup_ngrok(ngrok_token)

        # Kill existing
        os.system("pkill -f streamlit >/dev/null 2>&1")
        time.sleep(2)

        # Environment
        env = os.environ.copy()
        env['JWT_SECRET'] = JWT_SECRET
        if email_password:
            env['EMAIL_APP_PASSWORD'] = email_password

        # Start Streamlit
        process = subprocess.Popen(
            ["streamlit", "run", "app.py", "--server.port", str(PORT), "--server.headless", "true"],
            env=env
        )

        time.sleep(3)

        # Try to create tunnel
        try:
            public_url = ngrok.connect(PORT).public_url
            print(f"\n🚀 App is running at: {public_url}")
        except:
            print(f"\n🚀 App is running at: http://localhost:{PORT}")
            print("⚠️ Ngrok tunnel failed - local only")

        print("\nPress ENTER to stop...")
        input()

    except Exception as e:
        print(f"Error: {e}")
    finally:
        if process:
            process.terminate()
        try:
            ngrok.kill()
        except:
            pass

# ================= RUN =================
if __name__ == "__main__":
    print("\n🔧 Checking pyngrok version...")
    try:
        import pyngrok
        print(f"✅ pyngrok version: {pyngrok.__version__}")
    except:
        print("⚠️ pyngrok not found, installing...")
        os.system("pip install pyngrok")

    print("\n📋 Choose launch mode:")
    print("1. Interactive mode (recommended)")
    print("2. Quick launch with command line args")

    choice = input("\nEnter choice (1 or 2): ").strip()

    if choice == "2" and len(sys.argv) > 1:
        token = sys.argv[1]
        email_pwd = sys.argv[2] if len(sys.argv) > 2 else None
        quick_launch(token, email_pwd)
    else:
        main()


🔧 Checking pyngrok version...
✅ pyngrok version: 7.5.0

📋 Choose launch mode:
1. Interactive mode (recommended)
2. Quick launch with command line args
🚀 POLICYNAV PRO - NGROK LAUNCHER (FIXED VERSION)

📌 Step 1: Ngrok Authentication
📌 Get your token from: https://dashboard.ngrok.com/get-started/your-authtoken

📧 Step 2: Email Configuration (Optional)
📧 For OTP functionality, enter Gmail App Password
📧 Get it from: https://myaccount.google.com/apppasswords

🔧 Configuring Ngrok...
🔄 Stopping existing processes...
✅ Email OTP configured

🚀 Starting Streamlit server...
⏳ Waiting for Streamlit to start....... ✅
🔗 Creating Ngrok tunnel...

✅ APPLICATION STATUS

🌐 PUBLIC URL: https://equally-platinous-phillis.ngrok-free.dev
📱 LOCAL URL: http://localhost:8501


✅ OTP features are enabled

----------------------------------------
COMMANDS:
  q - Quit and stop server
  s - Show status
----------------------------------------
